### Phase 5: Static Code Feature Extraction (AST & Tokens)

We build a Python syntax-aware extractor using the `ast` and `tokenize` libraries to extract detailed program characteristics.

In [1]:
import ast
import builtins
import io
import json
import tokenize
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "Data" / "Processed").is_dir()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Run 01_data_understanding.ipynb first to create Data/Processed.")
PROCESSED_DIR = ROOT / "Data" / "Processed"
df_raw = pd.read_parquet(PROCESSED_DIR / "validated_energy_results.parquet")
RANDOM_SEED = 42
FEATURE_NAMES = [
    "loc", "token_count", "ast_node_count", "cyclomatic_complexity", "loop_count",
    "nested_loop_depth", "conditional_count", "function_count", "max_nesting_depth",
    "function_call_count", "comprehension_count", "sort_call_count",
    "recursion_detected", "import_count", "builtin_call_count",
]

def extract_static_features(code_str):
    code = code_str if isinstance(code_str, str) else ""
    result = {name: 0 for name in FEATURE_NAMES}
    result["parsed_successfully"] = False
    result["loc"] = sum(bool(line.strip()) and not line.lstrip().startswith("#") for line in code.splitlines())
    try:
        ignored = {tokenize.INDENT, tokenize.DEDENT, tokenize.ENDMARKER, tokenize.ENCODING, tokenize.NL, tokenize.COMMENT}
        result["token_count"] = sum(token.type not in ignored for token in tokenize.generate_tokens(io.StringIO(code).readline))
    except (tokenize.TokenError, IndentationError, SyntaxError):
        pass

    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", SyntaxWarning)
            tree = ast.parse(code)
    except (SyntaxError, ValueError, RecursionError):
        return result
    result["parsed_successfully"] = True
    nodes = list(ast.walk(tree))
    result["ast_node_count"] = len(nodes)
    result["loop_count"] = sum(isinstance(node, (ast.For, ast.While, ast.AsyncFor)) for node in nodes)
    result["conditional_count"] = sum(isinstance(node, (ast.If, ast.IfExp)) for node in nodes)
    function_names = {node.name for node in nodes if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))}
    calls_by_name = []
    result["function_count"] = len(function_names)
    result["function_call_count"] = sum(isinstance(node, ast.Call) for node in nodes)
    result["comprehension_count"] = sum(isinstance(node, (ast.ListComp, ast.DictComp, ast.SetComp, ast.GeneratorExp)) for node in nodes)
    result["sort_call_count"] = sum(
        isinstance(node, ast.Call) and (
            (isinstance(node.func, ast.Name) and node.func.id == "sorted")
            or (isinstance(node.func, ast.Attribute) and node.func.attr == "sort")
        )
        for node in nodes
    )
    imported = set()
    for node in nodes:
        if isinstance(node, ast.Import):
            imported.update(alias.name for alias in node.names)
        elif isinstance(node, ast.ImportFrom):
            imported.add(node.module or "")
        elif isinstance(node, ast.Call) and isinstance(node.func, ast.Name):
            calls_by_name.append(node.func.id)
    result["import_count"] = len(imported)
    result["builtin_call_count"] = sum(name in dir(builtins) for name in calls_by_name)
    result["recursion_detected"] = int(bool(function_names.intersection(calls_by_name)))
    result["cyclomatic_complexity"] = 1 + sum(
        isinstance(node, (ast.If, ast.IfExp, ast.For, ast.AsyncFor, ast.While, ast.ExceptHandler))
        for node in nodes
    ) + sum(max(0, len(node.values) - 1) for node in nodes if isinstance(node, ast.BoolOp))

    loop_depth = 0
    nesting_depth = 0
    stack = [(tree, 0, 0)]
    nesting_types = (ast.For, ast.AsyncFor, ast.While, ast.If, ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)
    loop_types = (ast.For, ast.AsyncFor, ast.While)
    while stack:
        node, parent_loops, parent_nesting = stack.pop()
        current_loops = parent_loops + int(isinstance(node, loop_types))
        current_nesting = parent_nesting + int(isinstance(node, nesting_types))
        loop_depth = max(loop_depth, current_loops)
        nesting_depth = max(nesting_depth, current_nesting)
        stack.extend((child, current_loops, current_nesting) for child in ast.iter_child_nodes(node))
    result["nested_loop_depth"] = loop_depth
    result["max_nesting_depth"] = nesting_depth
    return result


In [2]:
print(f"Extracting paired AST features for {len(df_raw):,} rows...")
df_features = df_raw.reset_index(drop=True).copy()
for version in ("v0", "v1"):
    code_column = f"code_{version}_no_empty_lines"
    extracted = pd.DataFrame(
        [extract_static_features(code) for code in df_features[code_column]],
        index=df_features.index,
    ).add_suffix(f"_{version}")
    df_features = pd.concat([df_features, extracted], axis=1)
print(f"Extracted {len(FEATURE_NAMES)} static features for each version.")


Extracting paired AST features for 21,428 rows...
Extracted 15 static features for each version.


### Phase 6 & 7: Pairwise Features and Targets Creation

We construct the differences (v1 - v0) and build the classification target (`energy_winner`) and regression targets (`energy_difference_v1_minus_v0`).

In [3]:
# Randomly reverse each pair's orientation, swapping every matching V0/V1 column together.
rng = np.random.default_rng(RANDOM_SEED)
swap_mask = pd.Series(rng.random(len(df_features)) < 0.5, index=df_features.index)
v0_bases = {column[:-3] for column in df_features.columns if column.endswith("_v0")}
v1_bases = {column[:-3] for column in df_features.columns if column.endswith("_v1")}
if v0_bases != v1_bases:
    raise ValueError(
        f"Unpaired version columns found: V0-only={sorted(v0_bases - v1_bases)}, "
        f"V1-only={sorted(v1_bases - v0_bases)}."
    )
paired_columns = [(f"{base}_v0", f"{base}_v1") for base in sorted(v0_bases)]
for column_v0, column_v1 in paired_columns:
    old_v0 = df_features.loc[swap_mask, column_v0].copy()
    df_features.loc[swap_mask, column_v0] = df_features.loc[swap_mask, column_v1].to_numpy()
    df_features.loc[swap_mask, column_v1] = old_v0.to_numpy()
df_features["was_swapped"] = swap_mask.to_numpy()
print(f"Swapped orientation for {int(swap_mask.sum()):,} of {len(df_features):,} pairs.")

# All directional quantities and targets are recomputed from the swapped measurements.
diff_feature_names = [f"{name}_delta" for name in FEATURE_NAMES]
for name in FEATURE_NAMES:
    df_features[f"{name}_delta"] = df_features[f"{name}_v1"] - df_features[f"{name}_v0"]
df_features["energy_difference_v1_minus_v0"] = df_features["result_energy_v1"] - df_features["result_energy_v0"]
df_features["time_difference_v1_minus_v0"] = df_features["result_time_v1"] - df_features["result_time_v0"]
df_features["memory_difference_v1_minus_v0"] = df_features["result_memory_v1"] - df_features["result_memory_v0"]
df_features["energy_tie"] = df_features["energy_difference_v1_minus_v0"].eq(0)
df_features["energy_winner"] = (df_features["result_energy_v1"] < df_features["result_energy_v0"]).astype("int8")
tied_pair_count = int(df_features["energy_tie"].sum())
print(f"Exact tied pairs (excluded from training): {tied_pair_count:,}")
display(df_features[["problem_id", "was_swapped", "energy_winner", "energy_tie", "energy_difference_v1_minus_v0"]].head())


Swapped orientation for 10,619 of 21,428 pairs.
Exact tied pairs (excluded from training): 18


,problem_id,was_swapped,energy_winner,energy_tie,energy_difference_v1_minus_v0
0,p03657,False,0,False,27.86
1,p02837,True,0,False,23.84
2,p03556,False,1,False,-154.81
3,p02408,False,0,False,23.82
4,p03290,True,0,False,66.61


### Phase 8 & 9: Leakage Prevention & Final ML Dataset Compilation

We assign a clean group identifier (`problem_id`) so that training, validation, and evaluation are structured correctly on unseen problems.

In [4]:
# Keep parseable, non-tied pairs only; problem_id remains the group key for GroupKFold.
df_features["group_id"] = df_features["problem_id"]
model_ready_mask = (
    df_features["parsed_successfully_v0"]
    & df_features["parsed_successfully_v1"]
    & ~df_features["energy_tie"]
)
df_ml_ready = df_features.loc[model_ready_mask].copy().reset_index(drop=True)
tied_pairs = df_features.loc[df_features["energy_tie"]].copy()

per_version_features = [f"{name}_{version}" for name in FEATURE_NAMES for version in ("v0", "v1")]
configurations = {
    "per_version": per_version_features,
    "differences": diff_feature_names,
    "combined": per_version_features + diff_feature_names,
}
print(
    f"Training-ready: {len(df_ml_ready):,}; ties: {tied_pair_count:,}; "
    f"unparseable non-ties excluded: {len(df_features) - len(df_ml_ready) - tied_pair_count:,}."
)

df_ml_ready.to_parquet(PROCESSED_DIR / "prepared_training_dataset.parquet", index=False)
tied_pairs.to_parquet(PROCESSED_DIR / "tied_pairs.parquet", index=False)
manifest = {
    "seed": RANDOM_SEED,
    "swap_probability": 0.5,
    "rows_after_cleaning": int(len(df_features)),
    "tied_pairs_excluded": tied_pair_count,
    "unparseable_pairs_excluded": int(len(df_features) - len(df_ml_ready) - tied_pair_count),
    "training_rows": int(len(df_ml_ready)),
    "target": "energy_winner (1 means current V1 uses less energy; 0 means current V0 uses less)",
    "group_column": "group_id",
    "configurations": configurations,
    "codebert_experiment": {
        "prepared_text_columns": ["code_v0_no_empty_lines", "code_v1_no_empty_lines"],
        "target": "energy_winner",
        "group_column": "group_id",
        "implemented": False,
        "note": "The paired text and grouped target are saved; a CodeBERT fine-tuning run requires torch and transformers.",
    },
}
(PROCESSED_DIR / "feature_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
(PROCESSED_DIR / "feature_engineering_summary.json").write_text(
    json.dumps({key: value for key, value in manifest.items() if key != "configurations"}, indent=2),
    encoding="utf-8",
)
print(f"Saved features and manifest under {PROCESSED_DIR}.")


Training-ready: 21,410; ties: 18; unparseable non-ties excluded: 0.
Saved features and manifest under d:\Habiba\Uni\W26\Bachelor Thesis\energy-aware-code-recommendation\Data\Processed.


In [5]:
print("Available feature configurations:")
for name, columns in configurations.items():
    print(f"{name}: {len(columns)} features")
display(df_ml_ready[["problem_id", "group_id", "energy_winner", "energy_difference_v1_minus_v0"]].head())


Available feature configurations:
per_version: 30 features
differences: 15 features
combined: 45 features


,problem_id,group_id,energy_winner,energy_difference_v1_minus_v0
0,p03657,p03657,0,27.86
1,p02837,p02837,0,23.84
2,p03556,p03556,1,-154.81
3,p02408,p02408,0,23.82
4,p03290,p03290,0,66.61
